# M06 — Ridership Target and Features

M04 ke cleaned bus aur rail datasets se separate model-ready tables banayenge. Model training is notebook ka hissa nahi hai.

**Prediction:** given route/station aur target date ke liye ridership class `Low`, `Normal` ya `High`. Class us entity ke **training-period** ridership se defined hai; ye absolute passenger count, crowding, ya vehicle allocation nahi hai.


## 1. Historical simulation assumptions

- Publication delay **90 days assume** karte hain; CTA ki actual delay/revisions is notebook ne verify nahi ki. Sirf 91 din purani rides feature mein use karte hain.
- Training data: **2022-01-01 to 2023-09-30**. October–December 2023 training aur evaluation rows mein nahi, lekin agar cutoff allow kare to purani history ki tarah use ho sakte hain.
- Validation: **2024**; test: **2025**. Thresholds sirf training period se fit honge aur fixed rahenge.
- Training labels inhi fixed thresholds ke against **retrospectively** banengi. Ye har historical din ke liye us waqt trained model hone ka claim nahi hai.
- M05 ne full period visualize kiya tha, isliye 2025 ko perfectly untouched test nahi keh sakte. CTA ke historical revisions ke point-in-time snapshots bhi available nahi hain.


## 2. Cleaned data load karo

Pehle M04 run karo. IDs text ki tarah rakhe jate hain aur datasets combine nahi hote.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

if Path("backend/data/processed").is_dir():
    data_folder = Path("backend/data/processed")
elif Path("data/processed").is_dir():
    data_folder = Path("data/processed")
else:
    data_folder = Path("../data/processed")

bus = pd.read_csv(data_folder / "bus_cleaned.csv", dtype={"route": str}, parse_dates=["date"])
rail = pd.read_csv(data_folder / "rail_cleaned.csv", dtype={"station_id": str}, parse_dates=["date"])
datasets = {
    "bus": bus.rename(columns={"route": "entity_id"}),
    "rail": rail.rename(columns={"station_id": "entity_id"}),
}
for name, df in datasets.items():
    print(f"{name}: {len(df):,} cleaned rows")


bus: 167,648 cleaned rows
rail: 209,472 cleaned rows


## 3. Training period se class thresholds

Har route/station ke training-period rides ki 1/3 aur 2/3 quantiles nikalte hain:

- `Low`: rides lower threshold se kam ya barabar.
- `Normal`: lower se zyada aur upper se kam ya barabar.
- `High`: upper se zyada.

Kam az kam **30 training rows** aur alag lower/upper thresholds chahiye. Ye academic design choices hain, CTA ke official demand levels nahi. In conditions ko fail karne wali entity ki rows exclude hongi; counts neeche hain.


In [2]:
train_end = pd.Timestamp("2023-09-30")
thresholds = {}

for name, df in datasets.items():
    if df.empty or df[["entity_id", "date", "rides"]].isna().any().any():
        raise ValueError(f"{name}: data empty hai ya essential values missing hain; M04 check karein.")
    if df.duplicated(["entity_id", "date"]).any():
        raise ValueError(f"{name}: duplicate entity/date mili; M04 check karein.")
    train = df.loc[df["date"] <= train_end]
    groups = train.groupby("entity_id")["rides"]
    limits = groups.quantile([1/3, 2/3]).unstack()
    limits.columns = ["lower", "upper"]
    limits["training_days"] = groups.size()
    eligible = (limits["training_days"] >= 30) & (limits["lower"] < limits["upper"])
    thresholds[name] = limits.loc[eligible].reset_index()
    print(f"{name}: training entities = {len(limits)}, eligible = {eligible.sum()}, excluded = {(~eligible).sum()}")


bus: training entities = 130, eligible = 129, excluded = 1
rail: training entities = 143, eligible = 141, excluded = 2


## 4. 91-day historical feature

Same route/station ki **exactly 91 calendar days pehle** ki rides ko `lag_91_rides` banate hain. Missing day ka matlab zero nahi: agar history nahi milti to us row ko baad mein exclude karenge. 90-day publication delay assumption ki wajah se 91 din purana value use kiya hai.


In [3]:
features = {}
for name, df in datasets.items():
    history = df[["entity_id", "date", "rides"]].copy()
    history["date"] = history["date"] + pd.Timedelta(days=91)
    history = history.rename(columns={"rides": "lag_91_rides"})
    features[name] = df.merge(history, on=["entity_id", "date"], how="left", validate="one_to_one")
    features[name] = features[name].merge(thresholds[name], on="entity_id", how="left", validate="many_to_one")
    print(f"{name}: missing 91-day history across all rows = {features[name]['lag_91_rides'].isna().sum():,}")


C:\Users\muham\AppData\Local\Temp\ipykernel_20312\4273644652.py:4: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  history["date"] = history["date"] + pd.Timedelta(days=91)


bus: missing 91-day history across all rows = 12,761


C:\Users\muham\AppData\Local\Temp\ipykernel_20312\4273644652.py:4: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  history["date"] = history["date"] + pd.Timedelta(days=91)


rail: missing 91-day history across all rows = 13,104


## 5. Date splits aur unusable rows

Training, validation aur test dates neeche diye hain. Gap ki rows prediction examples nahi banti, lekin eligible purani history ke taur par use ho sakti hain. Missing threshold/history counts overlap kar sakte hain.


In [4]:
prepared = {}
for name, df in features.items():
    df["split"] = np.select(
        [df["date"].between("2022-01-01", "2023-09-30"),
         df["date"].between("2024-01-01", "2024-12-31"),
         df["date"].between("2025-01-01", "2025-12-31")],
        ["train", "validation", "test"], default="gap")
    candidates = df.loc[df["split"] != "gap"].copy()
    missing_limits = candidates["lower"].isna()
    missing_history = candidates["lag_91_rides"].isna()
    print(f"\n{name}: gap/outside rows = {(df['split'] == 'gap').sum():,}")
    print(f"Missing thresholds = {missing_limits.sum():,}; missing history = {missing_history.sum():,}")
    print(f"Total excluded candidate rows = {(missing_limits | missing_history).sum():,}")
    prepared[name] = candidates.loc[~(missing_limits | missing_history)].copy()



bus: gap/outside rows = 10,458
Missing thresholds = 388; missing history = 12,621
Total excluded candidate rows = 12,890

rail: gap/outside rows = 13,156
Missing thresholds = 3,287; missing history = 13,104
Total excluded candidate rows = 16,118


## 6. Target labels aur calendar features

Current date ki `rides` sirf label banane ke liye use hoti hain. Saved model features mein **current rides nahi** hongi. `lag_91_rides` purani observed rides hain. Calendar fields target date se nikle hain.


In [5]:
model_data = {}
for name, df in prepared.items():
    df["target"] = np.select(
        [df["rides"] <= df["lower"], df["rides"] <= df["upper"]],
        ["Low", "Normal"], default="High")
    df["weekday"] = df["date"].dt.dayofweek  # Monday=0, Sunday=6
    df["month"] = df["date"].dt.month
    df["day_of_month"] = df["date"].dt.day
    df["is_weekend"] = (df["weekday"] >= 5).astype(int)
    df["lag_91_rides"] = df["lag_91_rides"].astype("int64")
    columns = ["entity_id", "date", "weekday", "month", "day_of_month", "is_weekend", "lag_91_rides", "target", "split"]
    model_data[name] = df[columns].copy()
    print(f"\n{name} — class counts by split:")
    display(pd.crosstab(df["split"], df["target"]).reindex(index=["train", "validation", "test"], columns=["Low", "Normal", "High"], fill_value=0))



bus — class counts by split:


target,Low,Normal,High
split,,,
train,18578,20144,23110
validation,8137,7753,25404
test,6536,6554,28084



rail — class counts by split:


target,Low,Normal,High
split,,,
train,22652,25039,29436
validation,12381,10574,28651
test,10490,8899,32076


## 7. Save model-ready data aur fixed thresholds

Output files `backend/data/processed/` mein save hongi aur dobara run karne par overwrite hongi. Train, validation aur test ki rows ek file mein `split` column se identify hoti hain. Threshold files future interpretation ke liye zaroori hain; inhein train period se hi calculate kiya gaya hai.

**ML ke waqt inputs:** `entity_id`, `weekday`, `month`, `day_of_month`, `is_weekend`, `lag_91_rides`. `target` answer hai. `date` split/inspection ke liye hai aur `split` training feature nahi hai. Encoding aur koi learned transformation sirf train set par fit hogi.


In [6]:
for name, df in model_data.items():
    if set(df["split"]) != {"train", "validation", "test"}:
        raise ValueError(f"{name}: ek ya zyada splits empty hain; output save nahi hoga.")

for name, df in model_data.items():
    data_path = data_folder / f"{name}_model_data.csv"
    threshold_path = data_folder / f"{name}_train_thresholds.csv"
    df.to_csv(data_path, index=False, date_format="%Y-%m-%d")
    thresholds[name].to_csv(threshold_path, index=False)
    print(f"Saved: {data_path} ({len(df):,} rows)")
    print(f"Saved: {threshold_path} ({len(thresholds[name]):,} entities)")


Saved: ..\data\processed\bus_model_data.csv (144,300 rows)
Saved: ..\data\processed\bus_train_thresholds.csv (129 entities)
Saved: ..\data\processed\rail_model_data.csv (180,198 rows)
Saved: ..\data\processed\rail_train_thresholds.csv (141 entities)


## Viva points aur limitations

- Labels route/station ke **training data** ki quantiles par based hain; saari entities ke liye same passenger-count boundary nahi.
- Exact 91-day lag se assumed cutoff follow hota hai. Missing history ko zero nahi banaya.
- 2024/2025 labels evaluation ke liye current actual rides se banti hain, lekin current rides model input nahi.
- Unknown/new entities, insufficient history, class imbalance aur equal thresholds ke results ko review karna hoga.
- 90-day delay assumption validate nahi hui; historical data revisions aur M05 mein full-period viewing ki wajah se ye perfect operational backtest ya untouched test nahi.
- No model, accuracy, tuning ya performance improvement is module mein claim nahi hota.
